##Import

In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve_triangular
import scipy.sparse as sp
import scipy.linalg as la

##Carica dati

In [4]:
# Caricamento del vettore rhs.txt
rhs = np.loadtxt("rhs.txt")
num_incognite = len(rhs)
print(f"Dimensioni del vettore rhs: {num_incognite}")

# Caricamento della matrice sparsa A.txt
dati_A = np.loadtxt("A.txt")
righe = dati_A[:, 0].astype(int)
colonne = dati_A[:, 1].astype(int)
valori = dati_A[:, 2]
print(f"Dimensioni della matrice A: {dati_A.shape}")

Dimensioni del vettore rhs: 16
Dimensioni della matrice A: (64, 3)


##COO CSC e Cholesky

In [5]:
# Costruzione della matrice sparsa A in formato COO
A_coo = sp.coo_matrix((valori, (righe, colonne)), shape=(num_incognite, num_incognite))

# Conversione della matrice A in formato CSC (Compressed Sparse Column)
A_csc = A_coo.tocsc()
print(f"Matrice A convertita con successo in formato CSC ({A_csc.shape[0]}x{A_csc.shape[1]}).")


# 2. Fattorizzazione di Cholesky: -A = L * L^T
print("\n--- 2. Fattorizzazione di Cholesky ---")
# Definiamo la funzione my_cholesky come indicato nelle specifiche
def my_cholesky(A):
    # -A è definita positiva
    neg_A = -A.toarray() if sp.issparse(A) else -A
    # lower=True restituisce la matrice triangolare inferiore L tale che -A = L @ L.T
    L = la.cholesky(neg_A, lower=True)
    # Ritorna come csc_matrix sparsa
    return sp.csc_matrix(L)

# Eseguiamo la fattorizzazione di Cholesky per ottenere la matrice L
L = my_cholesky(A_csc)
print("Fattorizzazione di Cholesky -A = L * L^T eseguita con successo.")
print(f"Dimensione del fattore L: {L.shape}")


Matrice A convertita con successo in formato CSC (16x16).

--- 2. Fattorizzazione di Cholesky ---
Fattorizzazione di Cholesky -A = L * L^T eseguita con successo.
Dimensione del fattore L: (16, 16)


## Risoluzione sistema lineare

-A * u = -rhs  =>  L * L^T * u = -rhs

In [6]:

# Nota: Dato che fattorizziamo -A, trasformiamo l'equazione A * u = rhs in (-A) * u = -rhs
b = -rhs

# Passaggio 3a: Risoluzione del sistema triangolare inferiore L * y = b
y = spsolve_triangular(L, b, lower=True)

# Passaggio 3b: Risoluzione del sistema triangolare superiore L^T * u = y
L_T = L.transpose().tocsc()
u = spsolve_triangular(L_T, y, lower=False)

print("Sistema risolto con successo!\n")


Sistema risolto con successo!



#Verifica soluzione e termine residuo

In [7]:
# 4. Verifica della soluzione ed Errore di Residuo

residuo = np.linalg.norm(A_csc.dot(u) - rhs)
print(f"Norma del residuo ||A*u - rhs||: {residuo:.2e}")

print("\nPrime componenti della soluzione approssimata u:")
for idx, val in enumerate(u[:min(9, len(u))]):
    print(f"  u[{idx}] = {val:.6f}")

Norma del residuo ||A*u - rhs||: 2.71e-15

Prime componenti della soluzione approssimata u:
  u[0] = 0.033333
  u[1] = 0.046667
  u[2] = 0.046667
  u[3] = 0.066667
  u[4] = 0.033333
  u[5] = 0.046667
  u[6] = 0.046667
  u[7] = 0.066667
  u[8] = 0.033333
